# Voz_Noslen F5-TTS ONNX (Modo Turbo) - v2026.06.19

Este notebook automatiza a criação do pacote **Turbo** para o F5-TTS, otimizado para CPU e Cloud Run.

### Estrutura do Pacote Gerado
```text
/onnx_package_turbo_TIMESTAMP/
├── onnx/f5_tts_transformer_core.onnx  # Grafo Turbo (x, cond -> dx)
├── model/vocab.txt                   # Dicionário de tokens
├── reference/referencia_voz.wav      # Áudio de referência
├── manifest.json                     # Versão e lista de arquivos
├── metadata.json                     # Contrato ONNX e Shapes
└── validation.json                   # Status da exportação e testes
```

**Requisitos:** Internet ativada no Kaggle.

In [ ]:
# 1) Instalação de dependências
!pip install -q huggingface_hub f5-tts safetensors onnx onnxscript onnxruntime vocos

In [ ]:
# 2) Criação do Script Packager Turbo
from pathlib import Path

script_content = r'''
from __future__ import annotations

import argparse
import json
import logging
import os
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urljoin, urlparse, urlsplit, urlunsplit

# Configurações de Caminhos e Versão
PACKAGER_VERSION = "2026.06.19.turbo.v3"
DEFAULT_SOURCE_URL = "https://huggingface.co/buckets/warllem/Voz_Noslen"
DEFAULT_VOICE_DIR = "voices/v_minha_voz_f5_tts_ptbr"

def resolve_work_root() -> Path:
    configured = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
    try:
        configured.mkdir(parents=True, exist_ok=True)
        return configured
    except OSError:
        return Path("/tmp/voz_noslen_turbo_working")

WORK_ROOT = resolve_work_root()
DOWNLOAD_DIR = WORK_ROOT / "turbo_source_snapshot"
STAGING_DIR = WORK_ROOT / "turbo_staging_area"
LOG_PATH = WORK_ROOT / "voz_noslen_turbo_packager.log"

@dataclass(frozen=True)
class TurboPaths:
    source: Path
    staging: Path
    onnx: Path
    model: Path
    reference: Path
    manifest: Path
    metadata: Path
    validation: Path

def setup_logging() -> logging.Logger:
    logger = logging.getLogger("turbo_packager")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
        logger.addHandler(handler)
    return logger

LOGGER = setup_logging()

# --- Utilitários de Arquivo ---

def clean_and_make_paths() -> TurboPaths:
    if STAGING_DIR.exists():
        shutil.rmtree(STAGING_DIR)
    
    paths = TurboPaths(
        source=DOWNLOAD_DIR,
        staging=STAGING_DIR,
        onnx=STAGING_DIR / "onnx",
        model=STAGING_DIR / "model",
        reference=STAGING_DIR / "reference",
        manifest=STAGING_DIR / "manifest.json",
        metadata=STAGING_DIR / "metadata.json",
        validation=STAGING_DIR / "validation.json"
    )
    
    for p in [paths.onnx, paths.model, paths.reference]:
        p.mkdir(parents=True, exist_ok=True)
    return paths

def copy_readonly(src: Path, dst: Path):
    """Copia arquivos garantindo que a origem não seja alterada."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# --- Wrapper de Exportação ---

def export_turbo_core(checkpoint_path: Path, vocab_path: Path, paths: TurboPaths, manifest_data: dict):
    import torch
    import gc
    from f5_tts.infer.utils_infer import load_model
    from hydra.utils import get_class

    class F5TTSTurboWrapper(torch.nn.Module):
        """Contrato Turbo: x, cond, text_ids, text_lengths, time_steps -> dx"""
        def __init__(self, model: Any) -> None:
            super().__init__()
            self.transformer = getattr(model, "transformer", model)

        def forward(self, x, cond, text_ids, text_lengths, time_steps):
            # DiT.forward espera (x, cond, text, time, ...). Passar
            # text_lengths como quinto argumento posicional faz o modelo
            # interpretá-lo como drop_audio_cond/mask e quebra audio_mask.sum(dim=1).
            length_anchor = text_lengths.to(dtype=x.dtype).reshape(1, 1, 1)
            x = x + length_anchor - length_anchor
            return self.transformer(x=x, cond=cond, text=text_ids, time=time_steps)

    # Configuração da arquitetura (F5-TTS v1 Base)
    arch_cfg = {
        "dim": 1024, "depth": 22, "heads": 16, "ff_mult": 2, "text_dim": 512,
        "text_mask_padding": True, "conv_layers": 4, "attn_backend": "torch"
    }
    
    onnx_file = paths.onnx / "f5_tts_transformer_core.onnx"
    LOGGER.info(f"Carregando modelo para exportação: {checkpoint_path.name}")
    
    from f5_tts.model import DiT
    model = load_model(
        DiT, arch_cfg, str(checkpoint_path), 
        mel_spec_type="vocos", vocab_file=str(vocab_path), device="cpu"
    )
    model.eval()
    
    wrapped = F5TTSTurboWrapper(model).eval()
    
    # Exemplo de entrada para o tracer (FP32)
    example_inputs = (
        torch.randn(1, 128, 100, dtype=torch.float32),       # x
        torch.randn(1, 128, 100, dtype=torch.float32),       # cond
        torch.randint(0, 100, (1, 64), dtype=torch.int64),   # text_ids
        torch.tensor([64], dtype=torch.int64),               # text_lengths
        torch.tensor([0.5], dtype=torch.float32),            # time_steps
    )

    LOGGER.info("Iniciando torch.onnx.export (Turbo Contract)...")
    torch.onnx.export(
        wrapped, example_inputs, str(onnx_file),
        input_names=["x", "cond", "text_ids", "text_lengths", "time_steps"],
        output_names=["dx"],
        dynamic_axes={
            "x": {1: "duration"}, "cond": {1: "duration"},
            "text_ids": {1: "text_len"}, "dx": {1: "duration"}
        },
        opset_version=17, do_constant_folding=True, dynamo=False
    )
    
    del model
    gc.collect()
    return onnx_file

# --- Geração de Metadados ---

def generate_metadata(paths: TurboPaths, source_info: dict):
    metadata = {
        "project": "Voz_Noslen Turbo",
        "contract": {
            "inputs": ["x", "cond", "text_ids", "text_lengths", "time_steps"],
            "outputs": ["dx"],
            "opset": 17,
            "dtype": "float32"
        },
        "shapes": {
            "x": [1, "duration", 100],
            "cond": [1, "duration", 100],
            "text_ids": [1, "text_len"],
            "text_lengths": [1],
            "time_steps": [1]
        },
        "engine": "ONNX Runtime CPU / Cloud Run Turbo Backend"
    }
    paths.metadata.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

def generate_manifest(paths: TurboPaths, source_info: dict, runtime_files: dict):
    manifest = {
        "name": f"Voz_Noslen_Turbo_{datetime.now(timezone.utc).strftime('%Y%m%d')}",
        "version": PACKAGER_VERSION,
        "backend": "turbo-onnx",
        "source_origin": source_info.get("repo_id", "HuggingFace Buckets"),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "files": runtime_files
    }
    paths.manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

def validate_package(paths: TurboPaths):
    import onnx
    import onnxruntime as ort
    import numpy as np

    report = {"status": "pending", "checks": {}}
    
    # 1. Check ONNX integrity
    try:
        model = onnx.load(str(paths.onnx / "f5_tts_transformer_core.onnx"))
        onnx.checker.check_model(model)
        report["checks"]["onnx_structure"] = "valid"
    except Exception as e:
        report["checks"]["onnx_structure"] = f"invalid: {str(e)}"

    # 2. ORT Smoke Test
    try:
        sess = ort.InferenceSession(str(paths.onnx / "f5_tts_transformer_core.onnx"), providers=["CPUExecutionProvider"])
        report["checks"]["onnxruntime_load"] = "success"
        
        # Teste de inferência dummy
        feeds = {
            "x": np.random.randn(1, 16, 100).astype(np.float32),
            "cond": np.random.randn(1, 16, 100).astype(np.float32),
            "text_ids": np.zeros((1, 16), dtype=np.int64),
            "text_lengths": np.array([16], dtype=np.int64),
            "time_steps": np.array([0.5], dtype=np.float32)
        }
        outputs = sess.run(None, feeds)
        report["checks"]["inference_smoke_test"] = "passed"
        report["status"] = "verified"
    except Exception as e:
        report["checks"]["inference_smoke_test"] = f"failed: {str(e)}"
        report["status"] = "failed"

    paths.validation.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report

# --- Orquestração Principal ---

def main():
    LOGGER.info(f"=== Voz_Noslen Turbo Packager {PACKAGER_VERSION} ===")
    
    # Simulação de localização de arquivos (No Kaggle isso viria do download)
    # Aqui assumimos que o download do bucket HF já foi feito pelo notebook
    source_root = DOWNLOAD_DIR
    paths = clean_and_make_paths()

    # Localização dos arquivos fonte (Read-Only)
    # Procurando em subpastas caso seja um bucket HF
    v_dir = source_root / DEFAULT_VOICE_DIR
    ckpt = next(v_dir.glob("model/model_*.pt"))
    vocab = v_dir / "model/vocab.txt"
    ref = v_dir / "data_reference/referencia_voz.wav"

    # 1. Copiar ativos fixos para o pacote
    copy_readonly(vocab, paths.model / "vocab.txt")
    copy_readonly(ref, paths.reference / "referencia_voz.wav")
    
    # 2. Exportar ONNX
    try:
        export_turbo_core(ckpt, vocab, paths, {})
    except Exception as e:
        LOGGER.error(f"Falha na exportação ONNX: {e}")
        sys.exit(1)

    # 3. Gerar Metadados e Manifest
    runtime_files = {
        "onnx": "onnx/f5_tts_transformer_core.onnx",
        "vocab": "model/vocab.txt",
        "reference": "reference/referencia_voz.wav"
    }
    generate_metadata(paths, {})
    generate_manifest(paths, {}, runtime_files)

    # 4. Validar
    report = validate_package(paths)
    LOGGER.info(f"Validação concluída: {report['status']}")

    # 5. Finalizar (O notebook fará o zip/upload)
    LOGGER.info(f"Pacote Turbo gerado em: {paths.staging}")

if __name__ == "__main__":
    main()
'''

Path("f5_tts_onnx_packager_kaggle.py").write_text(script_content, encoding="utf-8")
print("Script Turbo Packager criado com sucesso.")


In [ ]:
# 3) Download dos Ativos Originais (Read-Only)
import html
import json
import os
import re
import shutil
from pathlib import Path
from urllib.parse import quote, urlparse
from urllib.request import Request, urlopen

SOURCE_URL = "https://huggingface.co/buckets/warllem/Voz_Noslen"
VOICE_DIR = "voices/v_minha_voz_f5_tts_ptbr"
DOWNLOAD_PATH = Path("/kaggle/working/turbo_source_snapshot")

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        try:
            hf_token = user_secrets.get_secret("HF_TOKEN")
        except Exception:
            hf_token = user_secrets.get_secret("hf_token")
    except Exception as e:
        print(f"Aviso: Kaggle Secrets nao disponivel ou nao habilitado ({e}).")

if hf_token:
    print(f"[CHECK] Token HF encontrado (comprimento: {len(hf_token)}). Bucket: {SOURCE_URL}")
else:
    print(f"[INFO] Nenhum token HF encontrado. Se o bucket '{SOURCE_URL}' for privado, o download falhara.")
    print("DICA: Va em 'Add-ons' -> 'Secrets' no Kaggle e verifique se 'HF_TOKEN' esta marcado como habilitado para este notebook.")

def bucket_headers():
    headers = {"User-Agent": "voz-noslen-kaggle-bucket-downloader/1.0"}
    if hf_token:
        headers["Authorization"] = f"Bearer {hf_token}"
    return headers

def fetch_text(url):
    req = Request(url, headers=bucket_headers())
    with urlopen(req, timeout=120) as response:
        return response.read().decode("utf-8")

def parse_bucket_url(source_url):
    parsed = urlparse(source_url.rstrip("/"))
    parts = [p for p in parsed.path.split("/") if p]
    if len(parts) < 3 or parts[0] != "buckets":
        raise ValueError(f"SOURCE_URL precisa apontar para /buckets/<owner>/<bucket>: {source_url}")
    return f"{parts[1]}/{parts[2]}", f"{parsed.scheme}://{parsed.netloc}"

def list_bucket_items(bucket_id, base_url, prefix):
    tree_url = f"{base_url}/buckets/{bucket_id}/tree/{quote(prefix, safe='/')}"
    page = fetch_text(tree_url)
    match = re.search(r'data-target="BucketFileList" data-props="([^"]+)"', page)
    if not match:
        raise RuntimeError(f"Nao foi possivel ler a listagem do bucket em {tree_url}")
    props = json.loads(html.unescape(match.group(1)))
    return props.get("items", [])

def download_bucket_file(bucket_id, base_url, file_path, output_root):
    file_url = f"{base_url}/buckets/{bucket_id}/resolve/{quote(file_path, safe='/')}?download=true"
    target = output_root / file_path
    target.parent.mkdir(parents=True, exist_ok=True)
    req = Request(file_url, headers=bucket_headers())
    with urlopen(req, timeout=600) as response, target.open("wb") as out:
        shutil.copyfileobj(response, out)
    print(f"[OK] {file_path} -> {target}")

def download_bucket_prefix(source_url, prefix, output_root):
    bucket_id, base_url = parse_bucket_url(source_url)
    pending = [prefix.strip("/")]
    downloaded = 0
    while pending:
        current = pending.pop(0)
        for item in list_bucket_items(bucket_id, base_url, current):
            item_type = item.get("type")
            item_path = item.get("path")
            if not item_path:
                continue
            if item_type == "directory":
                pending.append(item_path)
            elif item_type == "file":
                download_bucket_file(bucket_id, base_url, item_path, output_root)
                downloaded += 1
    if downloaded == 0:
        raise RuntimeError(f"Nenhum arquivo baixado de {source_url}/tree/{prefix}")
    return downloaded

print(f"Baixando ativos do Hugging Face Storage Bucket: {SOURCE_URL}/tree/{VOICE_DIR}")
DOWNLOAD_PATH.mkdir(parents=True, exist_ok=True)
files_count = download_bucket_prefix(SOURCE_URL, VOICE_DIR, DOWNLOAD_PATH)
print(f"Download concluido em: {DOWNLOAD_PATH} ({files_count} arquivos)")


In [ ]:
# 4) Execução da Conversão e Empacotamento
!python f5_tts_onnx_packager_kaggle.py

In [ ]:
# 5) Compactação Final do Pacote
import json
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display
from datetime import datetime
from huggingface_hub import HfApi

STAGING_DIR = Path("/kaggle/working/turbo_staging_area")
ONNX_FILE = STAGING_DIR / "onnx/f5_tts_transformer_core.onnx"
VALIDATION_FILE = STAGING_DIR / "validation.json"
zip_name = f"voz_noslen_turbo_{datetime.now().strftime('%Y%m%d_%H%M')}"
zip_base = Path("/kaggle/working") / zip_name

if not STAGING_DIR.exists():
    raise RuntimeError("ERRO: Pasta de staging não encontrada.")
if not ONNX_FILE.exists():
    raise RuntimeError("ERRO: ONNX não foi gerado. A exportação falhou; não existe pacote final válido.")
if not VALIDATION_FILE.exists():
    raise RuntimeError("ERRO: validation.json não foi gerado. O pacote não foi validado.")

validation = json.loads(VALIDATION_FILE.read_text())
if validation.get("status") != "verified":
    raise RuntimeError(f"ERRO: pacote não validado: {validation}")

zip_path = Path(shutil.make_archive(str(zip_base), 'zip', STAGING_DIR))
print(f"PACOTE TURBO PRONTO: {zip_path}")

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        try:
            hf_token = secrets.get_secret("HF_TOKEN")
        except Exception:
            hf_token = secrets.get_secret("hf_token")
    except Exception as e:
        raise RuntimeError(f"HF_TOKEN não encontrado nos Secrets do Kaggle: {e}")
if not hf_token:
    raise RuntimeError("HF_TOKEN vazio. Habilite o secret HF_TOKEN no Kaggle antes de fazer upload.")

HF_UPLOAD_REPO_ID = os.getenv("HF_UPLOAD_REPO_ID", "warllem/Voz_Noslen_Turbo")
HF_UPLOAD_REPO_TYPE = os.getenv("HF_UPLOAD_REPO_TYPE", "model")
HF_UPLOAD_FOLDER = os.getenv("HF_UPLOAD_FOLDER", "turbo")
HF_PRIVATE_REPO = os.getenv("HF_PRIVATE_REPO", "1").lower() not in {"0", "false", "no"}

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_UPLOAD_REPO_ID,
    repo_type=HF_UPLOAD_REPO_TYPE,
    private=HF_PRIVATE_REPO,
    exist_ok=True,
)
remote_path = f"{HF_UPLOAD_FOLDER.rstrip('/')}/{zip_path.name}"
api.upload_file(
    path_or_fileobj=str(zip_path),
    path_in_repo=remote_path,
    repo_id=HF_UPLOAD_REPO_ID,
    repo_type=HF_UPLOAD_REPO_TYPE,
    commit_message=f"Upload pacote turbo {zip_path.name}",
)
repo_prefix = "" if HF_UPLOAD_REPO_TYPE == "model" else f"{HF_UPLOAD_REPO_TYPE}s/"
uploaded_url = f"https://huggingface.co/{repo_prefix}{HF_UPLOAD_REPO_ID}/blob/main/{remote_path}"
print(f"UPLOAD HUGGING FACE CONCLUIDO: {uploaded_url}")
display(FileLink(str(zip_path)))